# EmbeddingGemma 2：英語20万語を保持する日本語Adapter（CTC）

既存の `en.eg2m`（英語Reader + 約20万語テーブル）と `encoder.eg2m` を**書き換えません**。英語Readerを1度だけ通し、512次元Soft Tokensと英語Readerの768次元出力を小型Adapterに与えます。英語・日本語の候補は原英語blankを共有したCTCで選択します。

**新しいランタイムでも実行できます**：キャッシュの `train/index.jsonl`、`dev/index.jsonl` と対応する `.npy` が無い場合、⑤で日本語・英語FLEURSとEmbeddingGemma 2を取得して自動再生成します。既存の有効なキャッシュがあれば再利用します。Google Colabの `/content` はランタイム終了・再作成で消えるため、再利用したい場合はキャッシュの永続保存が必要です。学習済み `mix.eg2m` は使用しません。

**重要**：`reader.pt` と既存 `en.eg2m` の英語Reader重みはバイト単位で照合します。不一致なら学習を停止します。従来英語の推論 `-L en` は不変ですが、新しい `-L adapter` の英語WERは毎エポック評価し、劣化する候補を採用しません。実音声による評価が必要です。
**手順**：①設定 → ②依存 → ③コード展開 → ④英語モデル → ⑤キャッシュの自動復元 → ⑥日本語語彙 → ⑦英語Reader固定特徴 → ⑧Adapter学習 → ⑨C用出力。


In [ ]:
# @title ① 設定：T4 / L4 / A100 / CPU 自動判定
from pathlib import Path
import sys, os, subprocess, torch
BASE=Path('/content/eg2_mix')
TOOLS=BASE/'tools'
DATA=BASE/'dataset'
TRAIN_LIMIT_PER_LANGUAGE=500  # @param {type:'integer'}
DEV_LIMIT_PER_LANGUAGE=60  # @param {type:'integer'}
# 0 = 全FLEURS音声、500/60は最初の動作確認用。以前のキャッシュがある場合は再利用します。
MODEL=BASE/'source_encoder'     # 以前の⑦で使ったGoogleのテキスト埋め込みモデル
TRAIN_INDEX=BASE/'cache/train/index.jsonl'
DEV_INDEX=BASE/'cache/dev/index.jsonl'
EN_PKG=BASE/'models/en.eg2m'
JA_VOCAB=BASE/'ja_adapter_vocab.txt'
JA_TABLE=BASE/'ja_adapter_table.npy'
FROZEN_CACHE=BASE/'cache/en_original'
TRAINED=BASE/'trained_adapter'
OUTPUT=BASE/'models/ja_adapter.eg2a'
RANK=64  # @param {type:'integer'}
EPOCHS=12  # @param {type:'integer'}
BATCH_SIZE=2  # @param {type:'integer'}
MAX_JA_VOCAB=8000  # @param {type:'integer'}
EN_WER_TOLERANCE=0.01  # @param {type:'number'}
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE=='cuda':print('GPU:',torch.cuda.get_device_name(0))
else:print('CPU fallback: potentially slow; in Colab choose a T4 GPU')
assert RANK>=1 and RANK<=256 and EPOCHS>=1 and BATCH_SIZE>=1
print('Device:',DEVICE)


In [ ]:
# @title ② Colab依存パッケージをインストール（FLEURSの再生成もサポート）
subprocess.run([sys.executable,'-m','pip','install','-q',
    'transformers>=5.19.0','sentence-transformers>=6.1.0',
    'numpy','huggingface_hub','datasets','soundfile','scipy',
    'sudachipy','sudachidict_core','torchcodec'],check=True)


In [ ]:
# @title ③ Adapter学習ツールを展開（このノートブックに内蔵）
import base64,io,zipfile
BUNDLE="""..."""
BASE.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as z:
    for member in z.namelist():
        if not member.startswith('tools/') or '..' in member.split('/'):
            raise ValueError('unexpected embedded tool name')
        file=BASE/member;file.parent.mkdir(parents=True,exist_ok=True)
        file.write_bytes(z.read(member))
print('Updated:',len(z.namelist()),'tools at',TOOLS)


In [ ]:
# @title ④ 元の英語 en.eg2m と reader.pt を確認（キャッシュは次の⑤で自動復元）
from huggingface_hub import hf_hub_download
READER_PT=Path(hf_hub_download(repo_id='mlboydaisuke/EmbeddingGemma-2-Transcribe-Reader',filename='reader.pt'))
EN_PKG.parent.mkdir(parents=True,exist_ok=True)
if not EN_PKG.is_file():
    candidates=[BASE/'models-mix/en.eg2m',BASE/'models-en/en.eg2m',Path('/content/en.eg2m')]
    existing=next((p for p in candidates if p.is_file()),None)
    if existing:
        import shutil
        shutil.copy2(existing,EN_PKG)
    else:
        print('en.eg2m がありません。以前作成した元の英語モデル（約234MB）をアップロードしてください')
        from google.colab import files
        uploaded=files.upload()
        if 'en.eg2m' not in uploaded:
            raise FileNotFoundError('Select en.eg2m, not mix.eg2m')
        EN_PKG.write_bytes(uploaded['en.eg2m'])
for needed in (EN_PKG,READER_PT):
    if not needed.is_file():raise FileNotFoundError(f'Missing required original English file: {needed}')
print('EN model:',EN_PKG,EN_PKG.stat().st_size,'bytes')
print('Frozen pretrained English Reader:',READER_PT)
print('次の⑤で日英の音声キャッシュを確認し、不足時は自動生成します。')


In [ ]:
# @title ⑤ 音声キャッシュを自動復元／不足時に再生成（新規Colabランタイム対応）
# このセルは train/dev index.jsonl と参照先の512次元.npyを全件検査します。
# 正常なキャッシュがあればダウンロード・音声エンコードは行いません。
# 無い場合はFLEURS日本語/英語のWAVとEmbeddingGemma 2公式Encoderを取得し、⑧相当を実行します。
import torch,subprocess,sys
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Cache preparation device:',DEVICE,
      torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(CPU mode)')
cmd=[sys.executable,'-u',str(TOOLS/'eg2_adapter_recover.py'),
    '--base',str(BASE),'--train-limit',str(TRAIN_LIMIT_PER_LANGUAGE),
    '--dev-limit',str(DEV_LIMIT_PER_LANGUAGE),'--device',DEVICE,'--dtype','auto']
log=BASE/'adapter_cache_recovery.log'
BASE.mkdir(parents=True,exist_ok=True)
with log.open('w',encoding='utf8') as f:
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          text=True,encoding='utf8',errors='replace',bufsize=1)
    for line in proc.stdout:
        print(line,end='',flush=True)
        f.write(line);f.flush()
    code=proc.wait()
if code:
    raise RuntimeError(f'音声キャッシュの復元失敗 (exit={code})。上の根本エラーまたは {log} を確認してください')
assert TRAIN_INDEX.is_file() and DEV_INDEX.is_file(), 'Soft-token index files were not created'
print('Ready:',TRAIN_INDEX,DEV_INDEX)


In [ ]:
# @title ⑥ 日本語専用語彙＋768次元の日本語テーブルを生成
# 英語20万語はここで生成/変更しません。日本語だけです。
subprocess.run([sys.executable,str(TOOLS/'eg2_adapter_vocab.py'),
    '--train',str(TRAIN_INDEX),'--out',str(JA_VOCAB),
    '--max-vocab',str(MAX_JA_VOCAB)],check=True)
# 再実行時もJA_VOCABを変えるとTABLE/Adapterは再作成が必要
if not MODEL.is_dir():
    print('公式EmbeddingGemma 2がありません。⑤のキャッシュ復元セルを先に実行してください。')
    raise FileNotFoundError(MODEL)
subprocess.run([sys.executable,'-u',str(TOOLS/'eg2_ja_table.py'),
    '--model-dir',str(MODEL),'--vocab',str(JA_VOCAB),
    '--out',str(JA_TABLE),'--device',DEVICE,'--dtype','auto','--batch','8'],check=True)
print('Japanese classes:',len(JA_VOCAB.read_text(encoding='utf8').splitlines()))


In [ ]:
# @title ⑦ 英語Reader（固定）の出力・20万語のスコアをキャッシュ
# 一度だけ計算し、CTCの各エポックでは再使用。英語重みが異なれば停止します。
cmd=[sys.executable,'-u',str(TOOLS/'eg2_ja_adapter.py'),'prepare',
     '--train',str(TRAIN_INDEX),'--dev',str(DEV_INDEX),
     '--reader-pt',str(READER_PT),'--en-package',str(EN_PKG),
     '--out-dir',str(FROZEN_CACHE),'--device',DEVICE]
log=BASE/'adapter_prepare.log'
with log.open('w',encoding='utf8') as f:
    process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='');f.write(line);f.flush()
    ret=process.wait()
if ret:raise RuntimeError(f'English cache failed ({ret}); inspect {log}')


In [ ]:
# @title ⑧ 日本語Adapter＋日英共通CTCを学習（英語Readerは固定）
cmd=[sys.executable,'-u',str(TOOLS/'eg2_ja_adapter.py'),'train',
    '--train',str(TRAIN_INDEX),'--dev',str(DEV_INDEX),
    '--cache-dir',str(FROZEN_CACHE),'--reader-pt',str(READER_PT),
    '--en-package',str(EN_PKG),'--ja-vocab',str(JA_VOCAB),
    '--ja-table',str(JA_TABLE),'--out-dir',str(TRAINED),
    '--rank',str(RANK),'--epochs',str(EPOCHS),'--batch',str(BATCH_SIZE),
    '--en-wer-tolerance',str(EN_WER_TOLERANCE),'--device',DEVICE]
log=BASE/'adapter_train.log'
with log.open('w',encoding='utf8') as f:
    process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='');f.write(line);f.flush()
    ret=process.wait()
if ret:raise RuntimeError(f'JA Adapter training failed ({ret}); inspect {log}')


In [ ]:
# @title ⑨ AdapterをC用の ja_adapter.eg2a に変換してダウンロード
import json, zipfile
subprocess.run([sys.executable,str(TOOLS/'eg2_ja_adapter.py'),'export',
    '--checkpoint',str(TRAINED/'adapter.pt'),'--ja-vocab',str(JA_VOCAB),
    '--out',str(OUTPUT),'--force'],check=True)
subprocess.run([sys.executable,str(TOOLS/'eg2_bundle.py'),'--verify',str(OUTPUT)],check=True)
print('Output:',OUTPUT,OUTPUT.stat().st_size,'bytes')
print('English en.eg2m remains unmodified. Copy ja_adapter.eg2a alongside it.')
from google.colab import files
files.download(str(OUTPUT))


## Linux C側での利用

このZIPの修正版 `catseye/transcript` をビルドし、`encoder.eg2m` と元の `en.eg2m` と学習済み `ja_adapter.eg2a` を `models/` に並べます。

```bash
make
./cats -a eg2-native-asr -m models -L en -w english.wav       # オリジナル英語と同じ
./cats -a eg2-native-asr -m models -L adapter -w mixed.wav     # 新しい日英共通CTC
```

`-H 0.08` は日本語切り替えの慎重さ（ログit差）を調整する実験用設定です。英語WERの悪化を防ぐため、元の英語・日本語・混合音声を各50件程度で比較してください。**元の英語Readerの重みを変えないことは保証できますが、英語の最終認識WERはゲートの誤選択で悪化する可能性があります。**

なお、合成連結音声だけでなく、自然な日英混合音声の学習・検証が重要です。T4でも動きますが、⑥は英語20万語全件を照合するため時間が掛かります。
⑤は既存キャッシュがあれば再利用します。ランタイムを再起動した場合、WAVや公式モデルがなければ再取得が必要です。
